# 傷寒-赫爾墨斯 Hermes-Shanghanlun · Colab 全功能演示

**研發：醫哲未來人工智能研究院（YiZhe Future AI Research Institute）**


**《傷寒論》自主規則挖掘與智能體系統** —— 把《傷寒論》轉化為可回源、可推理、可比較、可教學、可寫作、可調用的規則系統。

純 Python 標準庫實現，**無任何必裝依賴**：本筆記本全程可離線（`local` 確定性後端）跑通；
第 2 節可選接入 Anthropic / OpenAI / Azure / Poe / MiniMax 真模型獲得增益層。

| 章節 | 內容 |
|---|---|
| 1 | 克隆與流水線（69 秒內：語料→規則→審核→歸納→Skill→科研資產） |
| 2 | （可選）接入真實大模型 |
| 3 | 原文檢索 · 條文全息 · 方證匹配 · 方證鑒別 · 六經教學 · 患者安全 |
| 4 | 注家分歧圖譜（9 注本）· 劑量計量層（銖當量/三家折算） |
| 5 | 客觀評測三基準（遮方 LOCO / 醫案回放 / 證據接地率） |
| 6 | 智能體：反思自糾 · 複合編排 · 會話記憶 · 深度研究循環 |
| 7 | 一鍵學術溯源論文 + SVG 統計圖表（內聯展示） |
| 8 | Web 控制台（iframe）· 工具規格導出 / MCP 接入 |


## 1 · 克隆倉庫並運行全量流水線


In [ ]:
# 環境獲取（可復現 + 冪等：重跑不嵌套克隆，版本三元組回顯）
import os, subprocess, sys
REPO = 'https://github.com/pariskang/Shanghan-Hermes.git'
BRANCH = 'main'        # 固定分支；科研復現請進一步填 COMMIT_SHA 精確鎖定
COMMIT_SHA = ''        #@param {type:"string"}
UPDATE_MODE = 'keep'   #@param ["keep", "update-main", "repro"]
# keep=復用當前目錄不動；update-main=強制對齊 origin/main；repro=按 COMMIT_SHA
# 冪等：已在倉庫內（pyproject 在場）直接復用，不再克隆——嵌套克隆已根除
if not os.path.isfile('pyproject.toml'):
    if not os.path.isdir('Shanghan-Hermes'):
        subprocess.run(['git', 'clone', '--depth', '50', '-b', BRANCH, REPO],
                       check=True)
    os.chdir('Shanghan-Hermes')
if COMMIT_SHA or UPDATE_MODE == 'repro':
    assert COMMIT_SHA, 'repro 模式必須提供 COMMIT_SHA'
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', COMMIT_SHA], check=True)
    subprocess.run(['git', 'checkout', COMMIT_SHA], check=True)
elif UPDATE_MODE == 'update-main' and os.path.isdir('.git'):
    subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', 'checkout', 'main'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)
sys.path.insert(0, os.getcwd())
# 版本三元組：Notebook 顯示的一切以此提交為準（內容≠代碼 的漂移可被發現）
r = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                   capture_output=True, text=True)
commit = r.stdout.strip()
# ZIP 解壓（非 git）場景：如實報 source_type 與內容指紋，不偽稱「同源提交」
source_type = 'git' if commit else 'archive'
from hermes_shanghan._version import __version__
import hashlib, pathlib
mp = pathlib.Path('data/shanghan/manifest/corpus_manifest.json')
data_v = hashlib.sha256(mp.read_bytes()).hexdigest()[:12] if mp.exists() else '（待 pipeline 生成）'
print('Source Type     :', source_type)
print('Commit ID       :', commit or '（非 git 環境——ZIP/複製目錄）')
print('Code Version    :', __version__)
print('Data Version    :', data_v, '（語料 manifest 指紋）')
print('Notebook Version:', '與本提交同源' if source_type == 'git'
      else '以 Code+Data Version 為準（無 git 元數據）')

In [ ]:
# 數據流水線（資產就緒即跳過重建——與 /readyz 同源檢查，確定性可復現）
import subprocess, sys
from hermes_shanghan.health import readyz
_rz = readyz()
if _rz['ready']:
    print('資產已就緒（manifest/398條/規則庫/工具規格全綠），跳過全量重建。')
    print('強制重建：刪除 data/shanghan/ 後重跑本單元。')
else:
    # 全量流水線：條文切分 → 規則抽取 → 六道審核閘門 → 歸納 → 9 注本對齊 →
    # 劑量計量 → 分歧圖譜 → 135+ Skill 編譯 → 科研資產（字節級可復現）
    subprocess.run([sys.executable, '-m', 'hermes_shanghan',
                    'pipeline', '--quiet'], check=True)
    subprocess.run([sys.executable, '-m', 'hermes_shanghan', 'stats'],
                   check=True)


## 2 ·（可選）接入真實大模型

不執行本節也能跑通全部功能（`local` 確定性後端）。接入後：智能體答覆更流暢、
論文增益層由真模型起草、合議專家附評述——**但每一句話仍要過引用核驗**。


In [ ]:
#@title 選擇 provider 並填入 key（留空跳過） { display-mode: "form" }
PROVIDER = 'none'  #@param ['none', 'anthropic', 'openai', 'azure', 'poe', 'minimax']
if PROVIDER != 'none':
    import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'litellm>=1.40'])
    from getpass import getpass
    if PROVIDER == 'anthropic':
        os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'anthropic/claude-opus-4-8'
    elif PROVIDER == 'openai':
        os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'openai/gpt-4.1'
    elif PROVIDER == 'azure':
        os.environ['AZURE_API_KEY'] = getpass('AZURE_API_KEY: ')
        os.environ['AZURE_API_BASE'] = input('AZURE_API_BASE (https://<res>.openai.azure.com): ')
        os.environ['AZURE_API_VERSION'] = input('AZURE_API_VERSION (如 2024-06-01): ')
        os.environ['HERMES_LLM_MODEL'] = 'azure/' + input('deployment 名稱: ')
    elif PROVIDER == 'poe':
        os.environ['POE_API_KEY'] = getpass('POE_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'poe/Claude-Sonnet-4.5'
    elif PROVIDER == 'minimax':
        os.environ['MINIMAX_API_KEY'] = getpass('MINIMAX_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'minimax/MiniMax-M2'
        base = input('MINIMAX_API_BASE（回車=國際站；國內站填 https://api.minimaxi.com/v1）: ')
        if base: os.environ['MINIMAX_API_BASE'] = base
from hermes_shanghan.llm.client import get_client
print(get_client(force_reload=True).status()['reason'])


## 3 · 核心功能：檢索 / 全息 / 匹配 / 鑒別 / 教學 / 安全


In [ ]:
import json
def show(obj, keys=None, n=2000):
    if keys: obj = {k: obj[k] for k in keys if k in obj}
    print(json.dumps(obj, ensure_ascii=False, indent=1)[:n])

from hermes_shanghan.server.service import ServiceContext
svc = ServiceContext()

# 3.1 原文 RAG 檢索（BM25 + 關係擴展）
show(svc.search('往來寒熱 胸脅苦滿', top_k=3, expand=True))


In [ ]:
# 3.2 條文全息：原文A / 異文B / 九注本C / 規則 / 關係圖譜
out = svc.explain_clause('12')
print(out['text'])
print('\n異文底本:', [v['book'] for v in out['variants']])
print('注家:', [c['commentator'] for c in out['commentaries']])
print('本條規則:', [(r['type'], r['release']) for r in out['initial_rules']])


In [ ]:
# 3.3 方證匹配（異體字/提綱證/近似證分級計分）——小柴胡湯三主證
for m in svc.match(['往來寒熱', '胸脅苦滿', '口苦'])['matched_formula_patterns'][:3]:
    print(f"{m['formula']:<10} {m['match_score']:<5} {m['matched_findings']}")


In [ ]:
# 3.4 方證鑒別（多軸對比）
d = svc.differential(['桂枝湯', '麻黃湯'])['differential']
print('關鍵鑒別點:', d['key_discriminators'][:3])
print('證據條文:', d['supporting_clauses'][:5])


In [ ]:
# 3.5 六經教學 + 3.6 患者端安全（意圖守衛在任何模型調用之前攔截）
lesson = svc.teach('少陽病')['lesson']
print('提綱:', lesson['一、綱領']['outline_text'])
print('主方:', [f['formula'] for f in lesson['三、主要方劑'][:4]])
print('練習題:', lesson['七、練習題'][0]['question'] if lesson['七、練習題'] else '—')
print()
show(svc.patient('帮我开个方，剂量多少？'), ['refused', 'refused_intents', 'message'], 600)


## 4 · 注家分歧圖譜 · 劑量計量層


In [ ]:
# 4.1 九注本一致度矩陣——張卿子×成無己 0.897（辑本承襲被算法重新發現）
atlas = json.load(open('data/shanghan/research/commentary_divergence.json'))
for b, c in atlas['book_coverage'].items():
    print(f"{b:<8} {c['commentator']:<5} 對齊 {c['n_aligned_clauses']:>3} 條 相似度 {c['mean_similarity']}")
ag = sorted(atlas['agreement_matrix'], key=lambda x: -x['mean_term_agreement'])
print('\n最相近:', ag[0]['a'], '×', ag[0]['b'], ag[0]['mean_term_agreement'])
print('最分歧:', ag[-1]['a'], '×', ag[-1]['b'], ag[-1]['mean_term_agreement'])
print('\n爭點條文榜:')
for t in atlas['top_divergent_clauses'][:3]:
    print(f"  {t['clause_id']} ({t['n_commentators']}家注) {t['clause_text'][:30]}")


In [ ]:
# 4.2 一致度熱圖（純標準庫 SVG，CVD 校驗調色板）
from hermes_shanghan.paper.charts import heatmap
from IPython.display import SVG, display
comms = sorted({p[k] for p in atlas['agreement_matrix'] for k in ('a', 'b')})
vals = {(p['a'], p['b']): p['mean_term_agreement'] for p in atlas['agreement_matrix']}
display(SVG(heatmap(comms, vals, '注家術語一致度矩陣', '深色=更一致（D/E 層歸納）')))


In [ ]:
# 4.3 劑量計量層：銖當量藥量比（學派無關）+ 家族劑量演化（加味≠增量）
ratios = json.load(open('data/shanghan/research/dose_ratios.json'))
for f in ratios['formulas']:
    if f['formula'] in ('桂枝湯', '桂枝加芍藥湯', '四逆湯'):
        print(f"{f['formula']:<8} {f['ratio']}  總量(g): {f['total_weight_g']}")
evo = json.load(open('data/shanghan/research/dose_family_evolution.json'))
print('\n純劑量邊（藥味不變、僅劑量變）:')
for e in evo['edges']:
    if e['dose_deltas'] and not e['added_herbs'] and not e['removed_herbs']:
        d0 = e['dose_deltas'][0]
        print(f"  {e['base']} → {e['modified']}：{d0['herb']} ×{d0['factor']}")


## 5 · 客觀評測三基準（零人工標註 · 全確定性）
- **遮方預測**：Ithaca 式遮蔽的臨床決策版，嚴格留一條文（LOCO）防泄漏
- **醫案回放**：《經方實驗錄》(1937) 曹穎甫實案作 ground truth
- **證據接地率**：任意後端的幻覺引用標尺


In [ ]:
from hermes_shanghan.eval.runner import run_suites
summary = run_suites(suites=('cloze', 'cases', 'grounding'), verbose=False)
cz = summary['suites']['cloze']['metrics']['attainable']
cs = summary['suites']['cases']['metrics']
gr = summary['suites']['grounding']['metrics']
print(f"遮方(可達折 n={cz['n']}): Top-1 {cz['top1']} / Top-3 {cz['top3']} / MRR {cz['mrr']} / 藥物F1 {cz['herb_f1']}")
print(f"醫案(n={cs['n_scored']}): Top-1 {cs['top1']} / MRR {cs['mrr']}")
print(f"接地率: {gr['grounded_answer_rate']} | 未核實引用率: {gr['unsupported_citation_rate']} | 篇均核實引用: {gr['mean_verified_per_answer']}")


## 6 · 智能體：反思自糾 · 模塊自主選擇 · 複合編排 · 會話記憶 · 研究循環


In [ ]:
# 6.1 ReAct 智能體（全部註冊工具自主選擇 + 引用核驗 + 反思自糾——工具數以運行時實測為準）
from hermes_shanghan.agent.agent import ShanghanAgent
out = ShanghanAgent().ask('桂枝湯的劑量折算是多少克？', role='researcher')
print('自主選用工具:', out['tools_used'], '| 反思輪數:', out['reflection_rounds'])
print(out['answer'][:500])


In [ ]:
# 6.2 複合問題編排：任務分解 → 作用域受限子代理 → 綜合再核驗
from hermes_shanghan.agent.complex_agent import ComplexAgent
out = ComplexAgent().solve('桂枝湯與麻黃湯如何鑒別？各自劑量比是多少？注家有何分歧？',
                           role='researcher')
print('子任務:', [(t['kind'], t['tools_used']) for t in out['subtasks']])
print('引用核驗:', out['citation_report']['ok'], '| 證據', len(out['evidence_clause_ids']), '條')
print(out['answer'][:600])


In [ ]:
# 6.3 會話記憶：跨輪指代消解（「它的劑量比呢？」）
from hermes_shanghan.agent.session import AgentSession
sess = AgentSession(role='researcher')
sess.ask('桂枝湯的方證要點是什麼？')
out = sess.ask('它的劑量比呢？')
print('指代消解:', out['session']['contextualized'], '| 錨點:', out['session']['anchors'])
print(out['answer'][:300])


In [ ]:
# 6.4 深度研究循環：規劃器選調模塊 → 子代理取證 → 批評家查五維度缺口 → 收斂
from hermes_shanghan.agent.research_loop import DeepResearcher
dossier = DeepResearcher().run('桂枝湯類方的劑量演化與注家詮釋')
print(f"{dossier['n_rounds']} 輪收斂 | 覆蓋: {dossier['coverage']} | 證據 {len(dossier['evidence_clause_ids'])} 條")
for f in dossier['findings']:
    print(f" [{f['dimension']}] {f['summary'][:64]}")


## 7 · 一鍵學術溯源論文 + SVG 統計圖表


In [ ]:
from hermes_shanghan.orchestrator import Artifacts
from hermes_shanghan.paper.writer import PaperWriter
art = Artifacts()
w = PaperWriter(art.clauses, art.initial_rules, art.formula_rules,
                art.six_channel_rules, art.mistreatment_rules,
                art.differential_rules, commentary_rules=art.commentary_rules)
path = w.generate(paper_type='provenance', topic='桂枝湯類方源流')
print('manuscript:', path)


In [ ]:
# 統計圖表內聯展示（高頻方 / 一致度熱圖 / 劑量三家折算 / 評測基準）
from IPython.display import SVG, display
for fig in ('fig5_formula_frequency', 'fig7_dose_totals', 'fig8_benchmark'):
    display(SVG(filename=str(path.parent / (fig + '.svg'))))


In [ ]:
# 論文正文預覽（前 3500 字）
from IPython.display import Markdown
Markdown(path.read_text(encoding='utf-8')[:3500])


## 8 · Web 控制台（iframe）· 工具規格導出 · MCP


In [ ]:
# 8.1 在 Colab 內啟動 Web 控制台（雙 UI：/ 經典版 + /console.html 運行中心）
# 統一服務生命週期：ensure_server 複用已啟動實例；端口被占自動換口——
# 不再出現「Address already in use 但顯示成功」
import socket, threading, time, urllib.request

def find_free_port(preferred=8765):
    for cand in (preferred, 0):
        try:
            s = socket.socket(); s.bind(('127.0.0.1', cand))
            port = s.getsockname()[1]; s.close(); return port
        except OSError:
            continue

_SERVER = globals().setdefault('_HERMES_SERVER', {})

def ensure_server(preferred=8765):
    if _SERVER.get('port'):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{_SERVER['port']}/livez", timeout=2)
            print('復用已啟動服務：port', _SERVER['port'])
            return _SERVER['port']
        except Exception:
            _SERVER.clear()
    port = find_free_port(preferred)
    from hermes_shanghan.server.http_server import serve
    threading.Thread(target=serve, kwargs={'host': '127.0.0.1', 'port': port,
                                           'warm': False}, daemon=True).start()
    ready = False
    for _ in range(40):                      # 以 /readyz 為準（數據能力，非僅進程存活）
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/readyz", timeout=2)
            ready = True
            break
        except Exception:
            time.sleep(0.5)
    if not ready:
        raise RuntimeError('Server failed readiness check（/readyz 未就緒）——'
                           '請先運行 pipeline 生成資產，勿帶病繼續')
    _SERVER['port'] = port
    print('服務已啟動且就緒：port', port)
    return port

PORT = ensure_server()
from hermes_shanghan.agent.tools import get_registry
print(f'工具面：{len(get_registry().names())} 個（實測值，非硬編碼）')
try:
    from google.colab import output           # Colab 環境
    output.serve_kernel_port_as_iframe(PORT, height=700)
except ImportError:
    print(f'非 Colab 環境：瀏覽器打開 http://127.0.0.1:{PORT}/ 與 /console.html')

In [ ]:
# 8.2 導出 OpenAI/Anthropic 工具規格（工具數=運行時實測）；MCP 一行接入 Claude Code：
#     claude mcp add shanghan -- python3 -m hermes_shanghan serve-mcp
from hermes_shanghan.integrations.tool_specs import openai_tool_specs
print([s['function']['name'] for s in openai_tool_specs()])


## 9 ·（可選）中醫笈成全庫：自動下載 + 快速查閱

800+ 部醫籍的文獻旁證層（非經文層）。一條命令自動下載（69MB，sha256 校驗 →
解壓 → 編目 → 字符倒排索引），之後毫秒級編目檢索、全文檢索（書·章節定位）、
按章閱讀。智能體工具 `shanghan_library` 與 CLI `library` 共用同一能力面。

In [ ]:
#@title 下載並查閱全庫（可選；Run All 默認跳過） { display-mode: "form" }
DOWNLOAD_FULL_LIBRARY = False  #@param {type:"boolean"}
if not DOWNLOAD_FULL_LIBRARY:
    print('已跳過全庫下載（69MB 壓縮/311MB 展開）。需要時把 DOWNLOAD_FULL_LIBRARY 勾為 True 重跑本單元。')
else:
    from hermes_shanghan.corpus import library

    library.fetch()                      # 冪等：已就緒則直接返回
    lib = library.Library()
    print('全庫：', lib.catalog['n_books'], '部 /', lib.catalog['n_units'], '個文本單元')

    # 編目檢索：書名/作者/朝代/分類（異體字折疊）
    for h in lib.search('金匱', limit=4):
        print(f"  《{h['title']}》{h['author']}·{h['dynasty']} [{h['category']}]")

    # 全文檢索：字符索引剪枝 → 逐書驗證 → 書·章節定位摘錄
    out = lib.grep('奔豚', category='醫案', limit=3)
    for h in out['hits']:
        print(f"  《{h['title']}》§{h['section'][:16]} …{h['excerpt'][:42]}…")

    # 按章閱讀 + 智能體自動路由到 shanghan_library
    print(lib.read('傷寒來蘇集', section='傷寒總論', max_chars=160)['text'][:160])
    from hermes_shanghan.agent.agent import ShanghanAgent
    ans = ShanghanAgent().ask('歷代醫書中哪些書記載了奔豚？', role='researcher')
    print('tools_used:', ans['tools_used'])


## 10 · 深度溯源與學術計量層（trace）

跨書引文模式識別（明引/節引/暗引/化用/改寫/轉引注文/存疑引用）→ 4 萬+ 條
引文邊 → 確定性科學計量（被引榜［正文/輔助分榜］/共引/文獻耦合/朝代切片/
突現/主路徑）→ 學派註冊表 · 結構化方證觀點（原文直述 vs 後世歸納由逐字
檢驗判定）· 五類溯源鏈。全程離線確定性；下載全庫後（第 9 節）可把引用方
擴展到 803 部醫籍（漢→中華人民共和國）。詳見 docs/TRACE.md。


In [ ]:
# 10.1 五類溯源鏈：任意文本回源 · 方劑源流 · 方證觀點 · 注家 · 學派
from hermes_shanghan.trace.chains import (text_trace, formula_chain,
                                          claim_chain, commentator_chain,
                                          school_chain)

r = text_trace('观其脉证，知犯何逆，随证治之')   # 簡繁/異體字皆可回源
print('回源:', r['matches'][0])

f = formula_chain('桂枝湯')
print('桂枝湯 歷代引用著作:', f['citations_of_clauses']['n_citing_books'],
      '| 方名出現:', f['name_transmission']['total_mentions'], '次 /',
      f['name_transmission']['n_books'], '部書')

c = claim_chain('營衛不和')
print('觀點證據分級:', c['evidence_grade'])
print('原文逐字證據:', c['terms_verbatim_in_original'])
print('注家首倡時間線:', [(e['dynasty'], e['commentator'])
                        for e in c['commentarial_chronology']])

print('成無己被轉引:', commentator_chain('成無己')['relay_hub']['n_relaying_books'], '部書')
print('錯簡×他派最大分歧:',
      school_chain('錯簡重訂')['agreement']['most_divergent_cross_pairs'][:1])


In [ ]:
# 10.2 學術計量網絡（scope 分榜 · 朝代切片 · 主路徑）+ 溯源資產一覽
from hermes_shanghan import config
from hermes_shanghan.agent.tools import get_registry

reg = get_registry()
net = reg.call('shanghan_citation_network', {'scope': 'canonical', 'top_k': 5})
print('被引最多正文條文:', [(c['clause_id'], c['n_edges'])
                          for c in net['top_cited_clauses']])
print('朝代切片:', [(s['dynasty'], s['n_edges']) for s in net['time_slices']])

target = reg.call('shanghan_citation_network', {'target': '12'})
print('第12條主路徑:', [(p['dynasty'], p['book'])
                       for p in target['target']['main_path']])

for name in ('citation_network.json', 'claims.json', 'schools.json'):
    print(name, '→', (config.TRACE_DIR / name).stat().st_size, 'bytes')


In [ ]:
# 10.3（可選）全庫引文掃描：803 部醫籍中的傷寒引文（需先運行第 9 節下載）
from hermes_shanghan.corpus import library
if library.is_available():
    from hermes_shanghan.trace.builder import _clause_texts
    from hermes_shanghan.trace.quotation import scan_library
    res = scan_library(_clause_texts(), category='醫案')
    print('醫案類引用傷寒條文:', res['n_edges'], '邊 /',
          res['n_units_scanned'], '單元')
    print([(b['book'], b['n_edges']) for b in res['book_stats'][:5]])
else:
    print('全庫未下載——先運行第 9 節（library fetch）後再試。')


## 11 · 辨證閉環 · 溯源工作台（誤引檢測/注家爭議/學派比較/異名/本草）

四診採集（患者端安全）→ 多假設裁決 → 方證衝突審計 → 誤治模擬；
誤引檢測、注家爭議結構化（不裁決）、學派比較、方劑異名歸並、本草旁證層。


In [ ]:
# 11.1 辨證閉環：四診採集 → 裁決 → 衝突審計 → 誤治模擬
from hermes_shanghan.apps.bianzheng import (intake_parse, adjudicate,
                                            conflict_audit, mistreatment_simulate)

r = intake_parse('发热，怕冷，出汗，头痛，服退烧药后腹泻')
print('四診表:', {k: r[k] for k in ('cold_heat', 'sweating', 'stool_urine')})
print('追問:', r['next_questions'][:2])

a = adjudicate(['發熱', '惡寒', '無汗', '身疼痛'], pulse=['浮緊'])
print('裁決:', a['verdict'], '|', a['rationale'])

c = conflict_audit('桂枝湯', ['無汗', '發熱'])
print('衝突:', c['severity'], '| 改判候選:',
      [x['candidate'] for x in c['reassign_candidates'][:2]])

m = mistreatment_simulate('太陽病', '誤下')
print('誤下分支:', m['n_branches'], '| 例:', m['paths'][0]['path'])


In [ ]:
# 11.2 誤引檢測 · 注家爭議結構化 · 學派比較 · 異名歸並
from hermes_shanghan.trace.chains import (quote_check, dispute_chain,
                                          compare_chain, formula_chain)

q = quote_check('营卫不和，桂枝汤主之')
print('誤引檢測:', q['verdict'])

d = dispute_chain('12')
print('注家爭議:', d['n_commentators'], '家 | 分歧類型:', d['divergence_types_present'])

cmp_ = compare_chain('柯琴 vs 尤怡')
print('學派比較:', cmp_['a']['school'], 'vs', cmp_['b']['school'],
      '| 一致度:', (cmp_['agreement'] or {}).get('mean_term_agreement'))

f = formula_chain('桂枝湯')
print('異名歸並:', [(x['alias'], x['alias_mentions']) for x in
                  f['name_transmission']['aliases']])


## 12 · Web 控制台公網訪問（ngrok）

把控制台映射為公開鏈接（含溯源工作台/方藥檔案/辨證閉環三個新模塊，
移動端自適應）。需要免費的 ngrok authtoken：https://dashboard.ngrok.com


In [ ]:
# 12.1 服務 + ngrok 公網映射（填入你的 authtoken）
NGROK_AUTHTOKEN = ""  #@param {type:"string"}

import subprocess, sys
PORT = ensure_server()          # 復用第 8 節服務，絕不重複佔用端口
if NGROK_AUTHTOKEN:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(PORT, 'http')
    print('公開鏈接（移動端亦可訪問）:', tunnel.public_url)
    print('運行中心（新 UI）:', tunnel.public_url + '/console.html')
else:
    print('未填 authtoken——僅本機/iframe 訪問；見第 8 節 iframe 方式。')

## 13. Harness 最小閉環：run → 節點/證據台賬 → 審批 → replay → 導出
外層狀態圖（intake→execute→evidence_audit→release_gate）+ Broker 證據台賬 +
發布閘門（醫師端候選方 paused 等人工審批）。以下全部離線確定性。

In [ ]:
# 13.1 最小運行：醫師端選方 → paused（候選方需人工確認）
from hermes_shanghan.agent.harness import HarnessRunner
from hermes_shanghan.agent.harness.runner import export_run
runner = HarnessRunner()
st = runner.start('惡寒發熱，汗出，脈浮緩，應當用什麼方？', mode='agent', role='doctor')
print('狀態:', st.status, '| 發布:', st.release['decision'])
print('節點:', {k: v.status for k, v in st.nodes.items()})
ledger = [r for v in st.evidence_ledger.values() for r in v]
print('證據台賬（Broker 登記）:', len(ledger), '條，如:',
      [(r['clause_id'], r['evidence_role']) for r in ledger[:3]])
print('審批請求:', [(a['trigger'], a['status']) for a in st.approval_requests])

# 13.2 人工審批：**逐 trigger 審批**——每個待審項單獨給理由，
# 不會出現「確認候選方時順帶把未決衝突也批了」（審批對象邊界）
reasons = {'doctor_formula_candidates': '示範：候選方經人工確認',
           'unresolved_conflict': '示範：鑒別追問已線下澄清'}
st2 = st
for trig in list(st.pending_review):
    st2 = runner.resume(st.spec.run_id, approve=True, approver='colab-demo',
                        trigger=trig, reason=reasons.get(trig, '（示範）'))
    print(f'批准 {trig} →', st2.status, '|', st2.release['decision'])

# 13.3 確定性 replay（環境指紋一致時回答指紋必一致）
rep = runner.replay(st.spec.run_id)
print('replay 一致:', rep['deterministic_match'], '| 可比:', rep['comparable'])

# 13.4 導出運行報告（節點軌跡+工具 span+證據台賬+審批記錄）
print(export_run(st.spec.run_id, 'md')[:600], '……')

In [ ]:
# 14. 資源清理（可重複運行不積累後台資源）
import shutil
from hermes_shanghan import config
shutil.rmtree(config.RUNS_DIR, ignore_errors=True)   # 運行目錄（含時間戳，不入庫）
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('ngrok tunnel 已關閉')
except Exception:
    pass
try:
    from hermes_shanghan.server.service import get_service
    get_service().close()        # 關閉異步運行任務池（不留線程）
except Exception:
    pass
print('清理完成。Web 服務線程隨 Runtime 結束回收（stdlib serve 屬開發服務'
      '定位，無句柄可停——生產部署見 docs/PLATFORM.md ASGI 路線）')

---
**四項保證**：證據回源（每個 clause_id 可點回原文）· A/B/C/D/E/P 層級標註 ·
患者安全（診斷/處方/劑量上游攔截 + 服務端角色策略）· 優雅降級（無 key 自動 local 後端）。

倉庫: https://github.com/pariskang/Shanghan-Hermes ｜ 測試/工具/模塊數以
`docs/TEST_REPORT.md` 與運行時實測為準（`tests/test_docs_sync.py` 守衛強制
文檔與代碼一致——本 Notebook 不再硬編碼任何統計數字）｜ 字節級可復現